# Iris 3.0 explicado paso a paso

Este notebook está basado en `iris3.0.ipynb`, pero con explicaciones simples para estudiar y exponer.

Meta del proyecto: clasificar flores Iris con una red neuronal simple y bien generalizada.

## 1) Importación de librerías

- **numpy**: trabaja con arreglos numéricos rápidos.
- **pandas**: carga y organiza datos en tablas.
- **sklearn**: divide datos, normaliza y calcula métricas.
- **tensorflow/keras**: crea y entrena la red neuronal.
- **matplotlib**: dibuja gráficas para entender resultados.

Se usan porque este proyecto necesita: preparar datos, entrenar modelo y evaluar su rendimiento.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, precision_recall_fscore_support

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping

np.random.seed(42)
tf.random.set_seed(42)

print("TensorFlow:", tf.__version__)

## 2) Carga del dataset

Este dataset tiene medidas de flores Iris. Cada fila es una flor y la columna `especie` indica la clase.

In [ ]:
df = pd.read_csv('dataset3_iris_fisher (1).csv')

print('Dimensiones (filas, columnas):', df.shape)
print('Columnas:', list(df.columns))
print('\nTipos de datos:')
print(df.dtypes)

df.head()

### ¿Qué contiene y cómo está organizado?

- Las columnas numéricas describen tamaño de sépalo y pétalo.
- `especie_id` es la clase en número (0, 1, 2).
- `especie` es la clase en texto (Setosa, Versicolor, Virginica).

Con esto sabemos que hay datos listos para clasificación supervisada.

## 3) Estadísticas descriptivas

Estas estadísticas sirven para conocer el comportamiento general de los datos:
- **mean**: promedio
- **std**: qué tanto se dispersan
- **min / max**: valores extremos

No es matemática complicada: solo ayuda a entender los rangos antes de entrenar.

In [ ]:
df.describe()

## 4) Análisis básico

Aquí revisamos 3 ideas clave:
- El dataset está balanceado (misma cantidad por clase).
- Las clases sí se diferencian en varias medidas.
- Versicolor y Virginica suelen ser más parecidas entre sí.

También mostramos solo correlaciones importantes (sin matriz completa).

In [ ]:
# Balance de clases
print('Conteo por especie:')
print(df['especie'].value_counts())
print('\nPorcentaje por especie:')
print((df['especie'].value_counts(normalize=True) * 100).round(2).astype(str) + '%')

In [ ]:
# Promedios por especie para ver diferencias
features = ['longitud_sepalo_cm', 'ancho_sepalo_cm', 'longitud_petalo_cm', 'ancho_petalo_cm']

means_by_species = df.groupby('especie')[features].mean().round(2)
print('Promedio por especie:')
display(means_by_species)

print('Idea clave: Versicolor y Virginica quedan cercanas en varias medidas de pétalo.')

In [ ]:
# Correlaciones clave (sin matriz completa)
corr = df[features].corr()
pairs = [
    ('longitud_petalo_cm', 'ancho_petalo_cm'),
    ('longitud_sepalo_cm', 'longitud_petalo_cm'),
    ('ancho_sepalo_cm', 'ancho_petalo_cm')
]

print('Correlaciones clave:')
for a, b in pairs:
    print(f'- {a} vs {b}: {corr.loc[a, b]:.3f}')

## 5) División de datos

Separamos en:
- **Train**: el modelo aprende.
- **Validation**: ajustamos y vigilamos si se sobreajusta.
- **Test**: evaluación final real.

`stratify` significa mantener la proporción de clases en cada conjunto.

In [ ]:
X = df[features].values
y = df['especie_id'].values

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.40, random_state=42, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print('Train:', X_train.shape[0], 'muestras')
print('Validation:', X_val.shape[0], 'muestras')
print('Test:', X_test.shape[0], 'muestras')

## 6) Normalización

`StandardScaler` convierte cada variable para que tenga:
- media cercana a 0
- desviación estándar cercana a 1

Esto ayuda a que la red neuronal entrene mejor y más estable.

**Data leakage**: pasa cuando el modelo ve información del test antes de tiempo.
Se evita haciendo `fit` solo con train y luego `transform` en val/test.

In [ ]:
scaler = StandardScaler()

# Fit SOLO con train (evita data leakage)
scaler.fit(X_train)

X_train_scaled = scaler.transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print('Media de train escalado (aprox 0):', X_train_scaled.mean(axis=0).round(4))
print('Std de train escalado (aprox 1):', X_train_scaled.std(axis=0).round(4))

## 7) One-Hot Encoding

Convertimos etiquetas numéricas a formato vector, por ejemplo:
- clase 0 -> `[1, 0, 0]`
- clase 1 -> `[0, 1, 0]`
- clase 2 -> `[0, 0, 1]`

Esto es útil porque la salida de la red usa 3 neuronas (una por clase).

In [ ]:
y_train_cat = to_categorical(y_train, num_classes=3)
y_val_cat = to_categorical(y_val, num_classes=3)
y_test_cat = to_categorical(y_test, num_classes=3)

print('Forma original y_train:', y_train.shape)
print('Forma one-hot y_train_cat:', y_train_cat.shape)
print('Ejemplo clase 0:', y_train_cat[y_train == 0][0])

## 8) Concepto: funciones de activación

- **ReLU**: deja pasar valores positivos y pone en 0 los negativos. Ayuda a aprender patrones no lineales.
- **Softmax**: convierte la salida final en probabilidades que suman 1.

En este proyecto: ReLU en capas ocultas, Softmax en la capa final.

## 9) Arquitectura del modelo

- `Dense`: capa totalmente conectada que aprende combinaciones de variables.
- `Dropout`: apaga neuronas al azar durante entrenamiento para evitar sobreajuste.

Modelo usado (simple): `4 -> 12 -> 6 -> 3`.

In [ ]:
model = Sequential([
    Input(shape=(4,)),
    Dense(12, activation='relu'),
    Dropout(0.3),
    Dense(6, activation='relu'),
    Dropout(0.3),
    Dense(3, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

model.summary()

## 10) Entrenamiento

- **Época (epoch)**: una pasada completa por los datos de entrenamiento.
- **Batch size**: cuántas muestras procesa antes de actualizar pesos.
- **EarlyStopping**: detiene entrenamiento si validación ya no mejora y recupera los mejores pesos.

In [ ]:
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

history = model.fit(
    X_train_scaled, y_train_cat,
    validation_data=(X_val_scaled, y_val_cat),
    epochs=100,
    batch_size=8,
    callbacks=[early_stopping],
    verbose=0
)

print('Epocas entrenadas:', len(history.history['loss']))

## 11) Evaluación

- **Accuracy**: porcentaje de aciertos totales.
- **Precision**: de lo que el modelo dijo que era clase X, cuánto acertó.
- **Recall**: de los casos reales de clase X, cuántos detectó.

El conjunto **test** es el más importante porque es la prueba final con datos no vistos.

In [ ]:
train_loss, train_acc = model.evaluate(X_train_scaled, y_train_cat, verbose=0)
val_loss, val_acc = model.evaluate(X_val_scaled, y_val_cat, verbose=0)
test_loss, test_acc = model.evaluate(X_test_scaled, y_test_cat, verbose=0)

print('Accuracy Train:', round(train_acc, 4))
print('Accuracy Validation:', round(val_acc, 4))
print('Accuracy Test:', round(test_acc, 4))

y_pred = np.argmax(model.predict(X_test_scaled, verbose=0), axis=1)

print('\nReporte de clasificacion (Test):')
print(classification_report(y_test, y_pred, target_names=['Setosa', 'Versicolor', 'Virginica'], digits=3))

In [ ]:
precision, recall, f1, support = precision_recall_fscore_support(y_test, y_pred)
classes = ['Setosa', 'Versicolor', 'Virginica']

for i, cls in enumerate(classes):
    print(f'{cls}: precision={precision[i]:.3f}, recall={recall[i]:.3f}, f1={f1[i]:.3f}, soporte={support[i]}')

In [ ]:
# Curvas de entrenamiento (simple)
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='train')
plt.plot(history.history['val_accuracy'], label='val')
plt.title('Accuracy')
plt.xlabel('Epoca')
plt.ylabel('Valor')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='train')
plt.plot(history.history['val_loss'], label='val')
plt.title('Loss')
plt.xlabel('Epoca')
plt.ylabel('Valor')
plt.legend()

plt.tight_layout()
plt.show()

## 12) Conclusión

Si el accuracy de train, validation y test es parecido, el modelo **generaliza bien**.
Cuando no hay brecha grande entre train y test, decimos que **no hay sobreajuste fuerte**.

Resumen simple:
- Se prepararon bien los datos (stratify + normalización correcta).
- Se entrenó una red pequeña y clara.
- El resultado final en test refleja desempeño real del modelo.